# 🧍 Proyecto Integrador — Estimación de Pose y Despliegue en HF Spaces

**Materiales desarrollados por Matías Barreto, 2026**  
**Tecnicatura Superior en Ciencias de Datos e IA, IFTS24**  
* **Nomenclatura Oficial:** Procesamiento Digital de Imágenes  
* **Nombre de Trabajo:** Laboratorio de Tecnologías de la Imagen Digital  

---

## El proyecto

Este cuaderno es diferente a los anteriores. No es un tutorial paso a paso: es un **proyecto integrador**.

Vamos a partir de un código base funcional para detectar pose corporal con MediaPipe, y desde ahí vamos a construir y desplegar una aplicación web completa. El producto final va a estar publicado en Hugging Face Spaces y versionado en un repositorio de GitHub.

**Al completar este proyecto vamos a haber:**

1. Explorado MediaPipe Pose — la tercera solución de detección que vemos en esta unidad (además de Face Mesh y Hands).
2. Adaptado código de Jupyter a un script `app.py` listo para producción.
3. Desplegado una aplicación de visión artificial accesible desde cualquier navegador.
4. Publicado el código fuente en un repositorio de GitHub.

> ◈ Este cuaderno usa el **Cheatsheet de HF Spaces** como referencia para el despliegue. Conviene tenerlo abierto en otra pestaña: `Extras/Guias/HuggingFace-Spaces/Cheatsheet_Desarrollo_Space.ipynb`

## Microglosario

| Término | Definición | Analogía |
|---|---|---|
| **Pose estimation** | Detección automática de la posición de las articulaciones del cuerpo en una imagen | Como cuando un entrenador marca con stickers los puntos clave del cuerpo de un atleta para analizar su técnica |
| **Keypoint / Punto clave** | Coordenada que representa una articulación o punto anatómico específico (nariz, hombro, rodilla...) | Como los pines de un maniquí articulado: cada uno representa una unión móvil |
| **Visibilidad** | Valor entre 0 y 1 que indica qué tan seguro está el modelo de que ese punto es visible en la imagen | Como la confianza con la que un médico marca un punto en una radiografía: 1.0 = certeza total, 0.0 = pura suposición |
| **`app.py`** | Script Python que contiene la lógica completa de la aplicación, listo para ejecutarse fuera de Jupyter | Como el plano de una casa: en Jupyter dibujamos bocetos, en `app.py` está el plano final para construir |
| **Space (HF)** | Servidor gratuito de Hugging Face que ejecuta y publica aplicaciones Gradio | Como un hosting web especializado en aplicaciones de IA: subís el código y ellos lo sirven al mundo |

## ✦ MediaPipe Pose: los 33 puntos del cuerpo

MediaPipe Pose detecta **33 puntos clave** distribuidos por todo el cuerpo. A diferencia de Face Mesh (478 puntos en el rostro) o Hands (21 puntos en la mano), Pose cubre el esqueleto completo con menos puntos pero mayor alcance anatómico.

```
                [0] nariz
                   |
         [12] ─────┼───── [11]     ← hombros
          |                 |
         [14]              [13]    ← codos
          |                 |
         [16]              [15]    ← muñecas


         [24] ─────────── [23]     ← caderas
          |                 |
         [26]              [25]    ← rodillas
          |                 |
         [28]              [27]    ← tobillos
```

*Nota: los índices siguen la convención MediaPipe — lado derecho de la persona en índices pares, lado izquierdo en impares.*

Cada punto tiene cuatro valores:

| Atributo | Tipo | Descripción |
|---|---|---|
| `x` | float 0–1 | Posición horizontal normalizada |
| `y` | float 0–1 | Posición vertical normalizada |
| `z` | float | Profundidad relativa (aproximada) |
| `visibility` | float 0–1 | Confianza de que el punto es visible |

**Casos de uso:** análisis de postura, entrenamiento deportivo, ergonomía en el trabajo, coreografías, fisioterapia.

## Paso 1 — Instalación

Las mismas herramientas que ya conocemos. Si ya las instalaron en este entorno, la celda termina rápido.

In [ ]:
!pip install gradio mediapipe opencv-python-headless numpy --quiet

import mediapipe as mp
import gradio as gr
import cv2
import numpy as np

version_mediapipe = mp.__version__
version_gradio    = gr.__version__
version_opencv    = cv2.__version__

print("✓ Entorno listo.")
print(f"  mediapipe  {version_mediapipe}")
print(f"  gradio     {version_gradio}")
print(f"  opencv     {version_opencv}")

## Código base — detector de pose

La siguiente celda contiene la función central del proyecto. Algunas líneas están completas; otras tienen un `# TODO` donde ustedes van a tener que escribir.

Lean la función completa antes de ejecutarla. Los `# TODO` son parte de la consigna — no los salteen.

In [ ]:
import mediapipe as mp
import numpy as np
import cv2

# ── Inicialización del detector ─────────────────────────────────────────

modulo_pose    = mp.solutions.pose
modulo_dibujo  = mp.solutions.drawing_utils
estilos_dibujo = mp.solutions.drawing_styles

# TODO: completá los parámetros del detector
# min_detection_confidence controla cuánta certeza exige el modelo para
# considerar válida una detección. Probá valores entre 0.3 y 0.9.
# min_tracking_confidence aplica cuando se procesa video (cuadro a cuadro).
detector_pose = modulo_pose.Pose(
    static_image_mode=True,
    min_detection_confidence=0.5,   # TODO: ajustá este valor y observá el efecto
    min_tracking_confidence=0.5
)

print("✓ Detector de Pose inicializado.")


# ── Función principal ────────────────────────────────────────────────────

def detectar_pose(imagen_entrada):
    """
    Recibe una imagen RGB como array NumPy.
    Detecta los 33 puntos clave del cuerpo con MediaPipe Pose.
    Devuelve la imagen anotada y un texto con información de los puntos.
    """
    # Procesamos la imagen con el detector
    resultado = detector_pose.process(imagen_entrada)

    # Trabajamos sobre una copia para no modificar la imagen original
    imagen_anotada = imagen_entrada.copy()

    # Si no se detectó ninguna pose, informamos y devolvemos la imagen sin cambios
    if resultado.pose_landmarks is None:
        mensaje = "No se detectó ninguna figura humana en la imagen."
        return imagen_anotada, mensaje

    # Dibujamos el esqueleto completo sobre la imagen
    modulo_dibujo.draw_landmarks(
        image=imagen_anotada,
        landmark_list=resultado.pose_landmarks,
        connections=modulo_pose.POSE_CONNECTIONS,
        landmark_drawing_spec=estilos_dibujo.get_default_pose_landmarks_style()
    )

    # Extraemos los 33 landmarks en una lista para analizarlos
    lista_landmarks = resultado.pose_landmarks.landmark

    # Accedemos a puntos específicos por su índice
    # Los índices se pueden consultar en: mp.solutions.pose.PoseLandmark
    punto_hombro_derecho  = lista_landmarks[12]
    punto_hombro_izquierdo = lista_landmarks[11]
    punto_cadera_derecha  = lista_landmarks[24]

    # TODO: accedé a dos puntos más que te parezcan interesantes
    # Ejemplo: punto_rodilla_derecha = lista_landmarks[26]
    # punto_X = lista_landmarks[?]
    # punto_Y = lista_landmarks[?]

    # Calculamos la distancia horizontal entre los dos hombros
    # (diferencia de coordenadas x normalizadas — entre 0.0 y 1.0)
    distancia_hombros = abs(punto_hombro_derecho.x - punto_hombro_izquierdo.x)
    distancia_hombros_redondeada = round(distancia_hombros, 3)

    # TODO: calculá otra métrica que se pueda extraer de los landmarks
    # Pistas: distancia vertical entre hombro y cadera, visibilidad de un punto,
    #         comparación de alturas entre dos articulaciones...
    # mi_metrica = ...

    # Armamos el texto informativo que va a mostrarse junto a la imagen
    linea_hombros    = f"Distancia entre hombros: {distancia_hombros_redondeada}"
    linea_visibilidad = f"Visibilidad hombro derecho: {round(punto_hombro_derecho.visibility, 2)}"
    linea_cadera      = f"Cadera derecha y={round(punto_cadera_derecha.y, 3)}"

    # TODO: agregá la línea de tu métrica al texto de salida
    texto_info = linea_hombros + "\n" + linea_visibilidad + "\n" + linea_cadera

    return imagen_anotada, texto_info


# ── Interfaz de prueba ───────────────────────────────────────────────────

interfaz_pose = gr.Interface(
    fn=detectar_pose,
    inputs=gr.Image(
        label="Fotografía",
        type="numpy"
    ),
    outputs=[
        gr.Image(label="Pose detectada"),
        gr.Textbox(label="Información de puntos clave")
    ],
    title="Detector de Pose — MediaPipe",
    description="Subí una imagen de una o más personas. El modelo va a detectar los 33 puntos del esqueleto.",
    flagging_mode="never"
)

interfaz_pose.launch()

## ✎ Consigna 1 — Exploración

Antes de pasar al deploy, tomense unos minutos para entender lo que el detector devuelve.

1. **Probá con distintas fotos.** ¿Qué pasa con una imagen donde la persona está de espaldas? ¿Y si hay varias personas? ¿Y con una foto de cuerpo entero vs. una de cintura para arriba?

2. **Cambiá `min_detection_confidence`.** Ponelo en `0.3` y en `0.9`. ¿En qué tipo de imágenes notás la diferencia? ¿Por qué creés que existe ese parámetro?

3. **Completá los `# TODO` de la función `detectar_pose`.** Elegí dos puntos anatómicos adicionales, calculá una métrica propia y agregala al texto de salida. No hay una respuesta correcta única — lo importante es que puedas justificar por qué esa métrica es útil.

## De Jupyter a `app.py`

Un cuaderno Jupyter es un excelente entorno de exploración, pero no es lo que esperan los servidores de producción. Para desplegar en Hugging Face Spaces necesitamos un script Python clásico: `app.py`.

La lógica es la misma que ya conocemos de la unidad anterior — **arquitectura de 3 capas**:

```
┌──────────────────────────────────────────┐
│  CAPA 1 — Data Layer                     │
│  Carga única del modelo en memoria       │
│  (se ejecuta una sola vez al iniciar)    │
└──────────────────┬───────────────────────┘
                   ↓
┌──────────────────────────────────────────┐
│  CAPA 2 — Business Logic                 │
│  Función que procesa cada imagen         │
│  (se llama cada vez que llega una foto)  │
└──────────────────┬───────────────────────┘
                   ↓
┌──────────────────────────────────────────┐
│  CAPA 3 — Presentation Layer             │
│  Interfaz Gradio declarada con Blocks    │
│  (define cómo se ve la app)              │
└──────────────────────────────────────────┘
```

> ◈ Para los detalles de git y deploy, usen el Cheatsheet:
> `Extras/Guias/HuggingFace-Spaces/Cheatsheet_Desarrollo_Space.ipynb`

La celda siguiente genera los archivos de la aplicación directamente desde el cuaderno.

In [ ]:
# Esta celda genera los archivos del proyecto en una carpeta local.
# Una vez generados, esa carpeta se sube a Hugging Face Spaces con git.

import os

# Nombre de la carpeta donde vamos a guardar los archivos del Space
# TODO: cambien 'mi-pose-app' por el nombre que quieran darle a su Space
NOMBRE_PROYECTO = "mi-pose-app"

os.makedirs(NOMBRE_PROYECTO, exist_ok=True)

print(f"✓ Carpeta creada: {NOMBRE_PROYECTO}/")


# ── app.py ───────────────────────────────────────────────────────────────

contenido_app = '''
# app.py — Detector de Pose con MediaPipe
# Estructura: 3 capas (Data Layer / Business Logic / Presentation Layer)

import mediapipe as mp
import gradio as gr
import numpy as np


# ─────────────────────────────────────────────────────────────────────────
# CAPA 1 — DATA LAYER
# El modelo se carga una sola vez cuando arranca la aplicación.
# Si lo cargáramos dentro de la función, cada request esperaría la carga.
# ─────────────────────────────────────────────────────────────────────────

modulo_pose    = mp.solutions.pose
modulo_dibujo  = mp.solutions.drawing_utils
estilos_dibujo = mp.solutions.drawing_styles

# TODO: completá los parámetros con los valores que encontraron en la exploración
detector_pose = modulo_pose.Pose(
    static_image_mode=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)


# ─────────────────────────────────────────────────────────────────────────
# CAPA 2 — BUSINESS LOGIC
# Toda la lógica de procesamiento vive acá, desacoplada de la interfaz.
# Si mañana cambian la UI de Gradio a otra tecnología, esta función no cambia.
# ─────────────────────────────────────────────────────────────────────────

def detectar_pose(imagen_entrada):
    # TODO: peguen aquí la función que terminaron en la Consigna 1
    # Asegúrense de que incluya los TODO que completaron (métricas propias, etc.)
    pass


# ─────────────────────────────────────────────────────────────────────────
# CAPA 3 — PRESENTATION LAYER
# La interfaz declara cómo se ve la app, sin lógica de negocio adentro.
# ─────────────────────────────────────────────────────────────────────────

with gr.Blocks(title="Detector de Pose") as aplicacion:

    gr.Markdown("## Detector de Pose corporal — MediaPipe")
    gr.Markdown(
        "Subí una imagen de una persona y el modelo va a detectar "
        "los 33 puntos clave del esqueleto corporal."
    )

    with gr.Row():
        # TODO: definí los componentes de entrada
        # Pista: gr.Image con type="numpy" y un label descriptivo
        entrada_imagen = None   # reemplazá None por el componente correcto

    with gr.Row():
        # TODO: definí los dos componentes de salida
        # Pista: imagen anotada + cuadro de texto con métricas
        salida_imagen = None    # reemplazá None por el componente correcto
        salida_texto  = None    # reemplazá None por el componente correcto

    boton_analizar = gr.Button("Analizar pose", variant="primary")

    boton_analizar.click(
        fn=detectar_pose,
        inputs=entrada_imagen,
        outputs=[salida_imagen, salida_texto]
    )


if __name__ == "__main__":
    aplicacion.launch()
'''

ruta_app = os.path.join(NOMBRE_PROYECTO, 'app.py')
with open(ruta_app, 'w', encoding='utf-8') as archivo_app:
    archivo_app.write(contenido_app)

print("✓ app.py generado.")
print("  Completá los TODO antes de hacer el deploy.")

In [ ]:
# Generamos el requirements.txt — lista de dependencias del proyecto
# Hugging Face Spaces lee este archivo para instalar lo que necesita

import os

# Definimos cada dependencia con su versión mínima
dependencias = [
    "gradio>=4.0.0",
    "mediapipe>=0.10.0",
    "opencv-python-headless>=4.8.0",
    "numpy>=1.24.0",
]

# Unimos todas las líneas con salto de línea
contenido_requirements = "\n".join(dependencias)

ruta_requirements = os.path.join(NOMBRE_PROYECTO, 'requirements.txt')
with open(ruta_requirements, 'w', encoding='utf-8') as archivo_req:
    archivo_req.write(contenido_requirements)

# Confirmamos el contenido generado
print("✓ requirements.txt generado:")
print()
for dependencia in dependencias:
    print(f"  {dependencia}")

# Mostramos los archivos que están listos para el deploy
print()
print("Archivos del proyecto:")
archivos_generados = os.listdir(NOMBRE_PROYECTO)
for nombre_archivo in archivos_generados:
    print(f"  {NOMBRE_PROYECTO}/{nombre_archivo}")

## ✎ Consigna 2 — La interfaz

El `app.py` que generamos tiene varios `# TODO` pendientes. La consigna es completarlos hasta tener una aplicación que corra sin errores.

**Pasos:**

1. Abrí `app.py` en VS Code (o cualquier editor).

2. **Completá la función `detectar_pose`** pegando la versión final que construiste en la Consigna 1 — con las métricas propias incluidas.

3. **Completá los componentes de la interfaz** (`entrada_imagen`, `salida_imagen`, `salida_texto`). Usen el Cheatsheet de Extras como referencia para ver los componentes disponibles.

4. **Probá la app localmente** desde la terminal:
   ```bash
   cd mi-pose-app
   python app.py
   ```
   Si abre el navegador y funciona, están listos para el deploy.

> ◈ **¿La función no devuelve lo que esperan?** Revisá que los componentes en `outputs=` coincidan exactamente con los valores que devuelve `detectar_pose` (imagen + texto, en ese orden).

## ✎ Consigna 3 — El despliegue

Con la app funcionando localmente, es momento de publicarla. El proceso completo está detallado en el Cheatsheet de Extras — acá va el resumen:

### En Hugging Face Spaces

1. Entrá a [huggingface.co/new-space](https://huggingface.co/new-space)
2. Elegí un nombre para el Space (puede coincidir con `NOMBRE_PROYECTO`)
3. Seleccioná **SDK: Gradio** y **Hardware: CPU free**
4. Seguí los comandos de git del Cheatsheet para vincular y subir los archivos:
   ```bash
   git init
   git add .
   git commit -m 'feat: detector de pose con MediaPipe'
   git remote add origin https://huggingface.co/spaces/TU_USUARIO/TU_SPACE
   git branch -M main
   git push -u origin main
   ```

### En GitHub

5. Creá un repositorio nuevo en [github.com/new](https://github.com/new)
6. Vinculá el mismo proyecto con un segundo remote:
   ```bash
   git remote add github https://github.com/TU_USUARIO/TU_REPO
   git push github main
   ```

> ◈ El Space en HF va a quedar público y accesible por URL. Compartí el link cuando esté desplegado.

## ✎ Para pensar

Una vez que la aplicación esté desplegada, respondé estas preguntas:

1. **Sobre el modelo:** MediaPipe Pose fue entrenado con millones de imágenes. Sin embargo, en algunas fotos falla o detecta puntos en posiciones incorrectas. ¿En qué tipo de imágenes notaste más errores? ¿A qué factores creés que se debe?

2. **Sobre la arquitectura:** El `app.py` separa la carga del modelo (Capa 1) de la función de procesamiento (Capa 2). ¿Qué pasaría si cargáramos el modelo *dentro* de `detectar_pose`, en lugar de hacerlo una sola vez al inicio? ¿Por qué eso sería un problema en producción?

3. **Sobre el deploy:** Comparando el flujo que siguieron hoy (Jupyter → `app.py` → HF Spaces + GitHub) con cómo venían trabajando, ¿qué ventajas concretas tiene este proceso? ¿Qué parte les resultó más difícil de entender o ejecutar?